In [1]:
import laspy
import numpy as np
import os
from pathlib import Path
from tqdm import tqdm
import json

In [2]:
INPUT_DIR = r"C:\Users\lukas\OneDrive\Dokumenty\GeoAI_MODULES\GeoAI_podyplomowe\4_podstawy_sztucznej_inteligencji\data\test_scenes_cutted"
OUTPUT_DIR = r"C:\Users\lukas\OneDrive\Dokumenty\GeoAI_MODULES\GeoAI_podyplomowe\4_podstawy_sztucznej_inteligencji\data\dataset_sem_seg\test"
SOURCE_LABEL_ATTRIBUTE = "classification"
TARGET_LABEL_ATTRIBUTE = "classification_mapped"

# LAS DATASET (UAV): https://zenodo.org/records/8287792  # Creative Commons Attribution 4.0 International (CC BY 4.0)
# https://arxiv.org/pdf/2309.01279
# 0: unclassified
# 1: low_vegetation
# 2: terrain
# 3: out-points
# 4: stem
# 5: live branches
# 6: woody_branches
CLASS_MAP = { 
    2: 0, # val
    3: 0, # val
    4: 1,  # stem
    6: 2,  # woody_branches
    5: 2,  # live_branches
}

CLUTTER_CLASS_ID = 0

In [3]:
def remap_labels_in_file(input_path, output_path, source_attr, target_attr, class_map):
    source_las = laspy.read(input_path)
    
    original_labels = source_las[source_attr]
    remapped_labels = np.full(original_labels.shape, CLUTTER_CLASS_ID, dtype=np.uint8)
    for original_id, new_id in class_map.items():
        mask = original_labels == original_id
        remapped_labels[mask] = new_id

    new_header = source_las.header.copy()
    new_header.add_extra_dim(laspy.ExtraBytesParams(name=target_attr, type=np.uint8))

    final_las = laspy.LasData(new_header)
 
    for dim_name in source_las.point_format.dimension_names:
        try:
            source_values = source_las[dim_name]
            setattr(final_las, dim_name, source_values)
        except Exception as e:
            print(f"  Warning: Could not copy dimension '{dim_name}'. Reason: {e}")

    final_las[target_attr] = remapped_labels
    final_las.write(output_path)
    return True
    

In [4]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
files_to_process = list(Path(INPUT_DIR).glob('*.las')) + list(Path(INPUT_DIR).glob('*.laz'))

if not files_to_process:
    print(f"ERROR: No patch files found in {INPUT_DIR}")
else:
    processed_count = 0
    for file_path in tqdm(files_to_process[::1], desc="Remapping labels"):
        try:
            output_filename = file_path.name
            output_path = os.path.join(OUTPUT_DIR, output_filename)
            
            success = remap_labels_in_file(
                file_path,
                output_path,
                SOURCE_LABEL_ATTRIBUTE,
                TARGET_LABEL_ATTRIBUTE,
                CLASS_MAP
            )
            if success:
                processed_count += 1
        except Exception as e:
            print(f"  Failed to process {file_path.name}: {e}")
    
    output_map = {"clutter": CLUTTER_CLASS_ID, **CLASS_MAP}
    map_path = os.path.join(OUTPUT_DIR, 'class_map.json')
    with open(map_path, 'w') as f:
        json.dump(output_map, f, indent=4)
        
    print(f"\nLabel remapping finished. Processed {processed_count} files.")

Remapping labels: 100%|██████████| 41/41 [00:03<00:00, 11.33it/s]


Label remapping finished. Processed 41 files.
